1 — Install Libraries

In [1]:
!pip install groq pymupdf python-docx pandas openpyxl google-api-python-client google-auth


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.6 MB/s eta 0:00:00


2 — Imports & Configuration

In [2]:
import os
import json
import fitz
import pandas as pd
from docx import Document
from groq import Groq
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2 import service_account

Configuration

In [3]:
client = ('GROQ_API_KEY')

CREDENTIALS_FILE = "service_account.json"             # rename your file to this
FOLDER_ID        = "your_google_drive_folder_id_here"
DOWNLOAD_FOLDER  = "downloaded_resumes"
OUTPUT_FILE      = "output/resume_data.xlsx"

os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)
os.makedirs("output", exist_ok=True)

4 — Google Drive Functions

In [4]:
def get_drive_service():
    SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
    creds = service_account.Credentials.from_service_account_file(
        CREDENTIALS_FILE, scopes=SCOPES)
    return build('drive', 'v3', credentials=creds)

def list_files(service, folder_id):
    query = f"'{folder_id}' in parents and trashed=false"
    results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    return results.get("files", [])

def download_file(service, file_id, file_name):
    path = os.path.join(DOWNLOAD_FOLDER, file_name)
    request = service.files().get_media(fileId=file_id)
    with open(path, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return path

4 — Text Extraction

In [5]:
def extract_text_pdf(path):
    doc = fitz.open(path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return text

def extract_text_docx(path):
    doc = Document(path)
    return "\n".join(para.text for para in doc.paragraphs)

def extract_text(path, mime_type):
    if "pdf" in mime_type or path.endswith(".pdf"):
        return extract_text_pdf(path)
    elif "word" in mime_type or path.endswith(".docx"):
        return extract_text_docx(path)
    return ""

 5 — Groq LLM Parsing

In [6]:
def parse_with_llm(resume_text):
    prompt = f"""
    You are an expert HR Resume screener for a Senior AI Architect position.
    Extract the required information into a strictly structured JSON object.

    Required Fields:
    - candidate_name, email, phone
    - total_experience_years (integer or float)
    - core_ai_skills (list from: ML, Deep Learning, NLP, GenAI, LLMs, RAG, Fine-tuning, Agents, PyTorch, LangChain, LlamaIndex)
    - cloud_and_mlops (list from: AWS, Azure, GCP, Docker, Kubernetes, MLflow, CI/CD, Pinecone, FastAPI)
    - architecture_leadership (list from: System Design, Microservices, Team Leadership, Solution Architecture, Cost Optimization)

    Resume text:
    {resume_text[:4000]}
    """
    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )
    return json.loads(completion.choices[0].message.content)

 6 — Main Pipeline

In [8]:
print("Connecting to Google Drive...")
service = get_drive_service()

print("Fetching files...")
files = list_files(service, FOLDER_ID)
print(f"Found {len(files)} files")

results = []
for file in files:
    name      = file["name"]
    mime_type = file["mimeType"]

    if not any(x in mime_type for x in ["pdf", "word"]) and \
       not name.endswith((".pdf", ".docx")):
        print(f"Skipping: {name}")
        continue

    print(f"Processing: {name}...")
    path = download_file(service, file["id"], name)
    text = extract_text(path, mime_type)

    if not text.strip():
        print(f"  ⚠ Could not extract text from {name}")
        continue

    try:
        data = parse_with_llm(text)
        data["file_name"] = name

        exp_years         = float(data.get("total_experience_years", 0))
        ai_skills_list    = data.get("core_ai_skills", [])
        mlops_skills_list = data.get("cloud_and_mlops", [])

        data["core_ai_skills"]         = ", ".join(ai_skills_list) if isinstance(ai_skills_list, list) else str(ai_skills_list)
        data["cloud_and_mlops"]        = ", ".join(mlops_skills_list) if isinstance(mlops_skills_list, list) else str(mlops_skills_list)
        data["architecture_leadership"] = ", ".join(data.get("architecture_leadership", [])) if isinstance(data.get("architecture_leadership", []), list) else str(data.get("architecture_leadership", ""))

        # ✅ FIXED — 3 to 6 years filter
        has_required_experience = exp_years >= 3.0
        has_matching_skills     = len(ai_skills_list) >= 1 or len(mlops_skills_list) >= 1

        data["is_qualified"] = "YES" if has_required_experience and has_matching_skills else "NO"

        results.append(data)
        print(f"  ✅ {data['candidate_name']} | Exp: {exp_years} yrs | Qualified: {data['is_qualified']}")

    except Exception as e:
        print(f"  ❌ Error: {name}: {e}")

Connecting to Google Drive...
Fetching files...
Found 5 files
Processing: pranjl.pdf...
  ✅ Pranjal PATIL | Exp: 11.0 yrs | Qualified: YES
Processing: priya.pdf...
  ✅ Priya Verma | Exp: 12.0 yrs | Qualified: YES
Processing: surabh.pdf...
  ✅ Saurabh Mungale | Exp: 8.0 yrs | Qualified: YES
Processing: nehal.pdf...
  ✅ Ankit Sharma | Exp: 9.0 yrs | Qualified: YES
Processing: shantanu.pdf...
  ✅ Rahul Deshmukh | Exp: 10.0 yrs | Qualified: YES


Excel File

In [9]:
if results:
    df = pd.DataFrame(results)
    df.to_excel(OUTPUT_FILE, index=False)
    print(f"\n🎉 Excel saved: {OUTPUT_FILE}")
else:
    print("\n⚠ No data compiled.")


🎉 Excel saved: output/resume_data.xlsx
